# Prerequisites

In [ ]:
import os
from dotenv import load_dotenv
from pymongo import MongoClient
load_dotenv()

In [ ]:
MONGODB_URI = os.getenv("MONGO_URI")
mongodb_client = MongoClient(MONGODB_URI)
mongodb_client.admin.command("ping")

# Loading the dataset

In [ ]:
import json

In [ ]:
with open("dataset\\clean_data\\S08\\articles.json", 'r') as f:
    articles = json.load(f)

In [ ]:
len(articles)

In [ ]:
articles[0]

In [ ]:
with open("dataset\\clean_data\\S08\\questions.json", 'r') as f:
    questions = json.load(f)

In [ ]:
len(questions)

In [ ]:
questions[0]

# Chunking and embedding articles

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from typing import Dict, List
from voyageai.client import Client as VoClient
from tqdm import tqdm

In [ ]:
text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(model_name="gpt-4", chunk_size=200, chunk_overlap=0)

In [ ]:
def get_chunks(doc: Dict, text_field: str) -> List[str]:
    text = doc[text_field]
    chunks = text_splitter.split_text(text)
    return chunks

In [ ]:
vo = VoClient()

In [ ]:
def get_embeddings(content: List[str], input_type: str):
    embds_obj = vo.contextualized_embed(inputs=[content], model='voyage-context-3', input_type=input_type)
    if input_type == "document":
        embeddings = [emb for r in embds_obj.results for emb in r.embeddings]
    if input_type == "query":
        embeddings = embds_obj.results[0].embeddings[0]
    return embeddings

In [ ]:
embedded_articles = []
for article in tqdm(articles):
    chunks = get_chunks(article, 'body')
    chunk_embeddings = get_embeddings(chunks, 'document')
    for chunk, embedding in zip(chunks, chunk_embeddings):
        article_chunk = article.copy()
        article_chunk['body'] = chunk
        article_chunk['embedding'] = embedding
        embedded_articles.append(article_chunk)

In [ ]:
len(embedded_articles)

In [ ]:
embedded_articles[0]

# Ingesting data into MongoDB database

In [ ]:
DB_NAME = 'rag_discord'
COLLECTION_NAME = 'articles'
ATLAS_VECTOR_SEARCH_INDEX_NAME = 'vector_index'

In [ ]:
collection = mongodb_client[DB_NAME][COLLECTION_NAME]

collection.delete_many({})

In [ ]:
collection.insert_many(embedded_articles)
print(f"Ingested {collection.count_documents({})} documents into the {COLLECTION_NAME} collection.")

# Creating a vector search index

In [ ]:
from utils import create_index, check_index_ready

In [ ]:
model = {
    "name": ATLAS_VECTOR_SEARCH_INDEX_NAME,
    "type": "vectorSearch",
    "definition": {
        "fields": [
            {
                "type": "vector",
                "path": "embedding",
                "numDimensions": 1024,
                "similarity": "cosine"
            }
        ]
    }
}

In [ ]:
create_index(collection, ATLAS_VECTOR_SEARCH_INDEX_NAME, model)

In [ ]:
check_index_ready(collection, ATLAS_VECTOR_SEARCH_INDEX_NAME)

# Performing vector search

In [ ]:
def vector_search(user_query: str) -> List[Dict]:
    query_embedding = get_embeddings([user_query], "query")
    pipeline = [
        {
            "$vectorSearch": {
                "index": ATLAS_VECTOR_SEARCH_INDEX_NAME,
                "queryVector": query_embedding,
                "path": "embedding",
                "numCandidates": 20,
                "limit": 5
            }
        },
        {
            "$project": {
                "_id": 0,
                "topic": 1,
                "body": 1,
                "score": {"$meta": "vectorSearchScore"}
            }
        }
    ]
    
    results = collection.aggregate(pipeline)
    return results.to_list()

In [ ]:
questions[5]

In [ ]:
vector_search(questions[5])

# Implementing RAG

In [ ]:
import requests

In [ ]:
def create_prompt(user_query: str) -> str:
    context = vector_search(user_query)
    context = "\n\n".join([article.get("body", "") for article in context])
    prompt = f"Answer the question based only on the following context. If the context is empty, say I DON'T KNOW\n\nContext:\n{context}\n\nQuestion:{user_query}"
    return prompt

In [ ]:
def generate_answer(user_query: str) -> None:
    prompt = create_prompt(user_query)
    messages = [{"role": "user", "content": prompt}]
    